# Hybrid (CodeBERT + GraphCodeBERT) â€” multilang training (Java + C#) on Colab

Trains the hybrid model (frozen CodeBERT + GraphCodeBERT backbones, trained
linear head) on the real Juliet Java and C# splits by running
`src/models/train_hybrid_multilang.py` for each language. This is the last
untrained piece â€” RF/CodeBERT/GraphCodeBERT already have Java/C# numbers,
Hybrid never did.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine).

**Before running:** make sure `hybrid_multilang_bundle.zip` exists at the
repo root (bundles the script + the 6 required parquet files, ~34 MB).
Recreate it from the repo root if needed:

```python
import zipfile, os
files_to_zip = [
    "src/models/train_hybrid_multilang.py",
    "data/processed/multilang/java/train.parquet",
    "data/processed/multilang/java/val.parquet",
    "data/processed/multilang/java/test.parquet",
    "data/processed/multilang/csharp/train.parquet",
    "data/processed/multilang/csharp/val.parquet",
    "data/processed/multilang/csharp/test.parquet",
]
with zipfile.ZipFile("hybrid_multilang_bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files_to_zip:
        z.write(f, arcname=f)
```

## Step 1 â€” upload hybrid_multilang_bundle.zip

Run this cell, then click **Choose Files** and select `hybrid_multilang_bundle.zip`.

In [ ]:
from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 1, "Upload exactly one file: hybrid_multilang_bundle.zip"
zip_name = next(iter(uploaded))
print("Uploaded:", zip_name, f"({len(uploaded[zip_name]) / 1e6:.1f} MB)")

## Step 2 â€” unpack into the session's local disk

In [ ]:
import zipfile, os

WORK_DIR = "/content/work"
os.makedirs(WORK_DIR, exist_ok=True)

with zipfile.ZipFile(zip_name) as z:
    z.extractall(WORK_DIR)

os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())

## Step 3 â€” preflight: verify files unpacked correctly

In [ ]:
required = [
    "src/models/train_hybrid_multilang.py",
    "data/processed/multilang/java/train.parquet",
    "data/processed/multilang/java/val.parquet",
    "data/processed/multilang/java/test.parquet",
    "data/processed/multilang/csharp/train.parquet",
    "data/processed/multilang/csharp/val.parquet",
    "data/processed/multilang/csharp/test.parquet",
]

missing = [p for p in required if not os.path.isfile(p)]

for p in required:
    ok = os.path.isfile(p)
    size = f"{os.path.getsize(p) / 1e6:.1f} MB" if ok else "MISSING"
    print(f"{'OK ' if ok else 'XX '} {p}  {size}")

assert not missing, (
    f"{len(missing)} required file(s) missing after unzip -- rebuild "
    "hybrid_multilang_bundle.zip and re-upload."
)

## Step 4 â€” install dependencies

In [ ]:
!pip install -q transformers datasets scikit-learn pandas pyarrow tqdm

Same torchvision/datasets incompatibility hit during the Java/C# CodeBERT/GraphCodeBERT runs. Not needed for text classification.

In [ ]:
!pip uninstall -y -q torchvision torchaudio

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (training will be slow -- enable a GPU runtime)")

## Step 5 â€” train on Java

Uses `subprocess.run(..., check=True)` so a failing script raises a real exception with the full traceback instead of silently reporting success.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "src/models/train_hybrid_multilang.py", "--language", "java"],
    check=True,
)
print("java training finished with exit code", result.returncode)

### Safety net — download Hybrid (Java) now

Run this immediately after the step above finishes, before starting the next language. If the session dies or runs out of compute later, you keep this result instead of losing the whole notebook's progress.

In [ ]:
import os, zipfile

_ckpt = "experiments/hybrid-java"
_res = "results/hybrid_java_evaluation.json"
_name = "safety_hybrid_java"
if os.path.isdir(_ckpt):
    exclude = {"optimizer.pt", "scheduler.pt", "rng_state.pth"}
    with zipfile.ZipFile(f"{_name}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _dirs, fnames in os.walk(_ckpt):
            for fn in fnames:
                if fn in exclude:
                    continue
                fp = os.path.join(root, fn)
                zf.write(fp, os.path.relpath(fp, os.path.dirname(_ckpt)))
    print(f"{_name}.zip -> {os.path.getsize(_name + '.zip') / 1e6:.1f} MB")
    files.download(f"{_name}.zip")
else:
    print("No checkpoint dir at", _ckpt, "-- check the training step above succeeded.")
if os.path.isfile(_res):
    files.download(_res)
else:
    print("No results file at", _res, "-- check the training step above succeeded.")


## Step 6 â€” train on C#

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "src/models/train_hybrid_multilang.py", "--language", "csharp"],
    check=True,
)
print("csharp training finished with exit code", result.returncode)

### Safety net — download Hybrid (C#) now

Run this immediately after the step above finishes, before starting the next language. If the session dies or runs out of compute later, you keep this result instead of losing the whole notebook's progress.

In [ ]:
import os, zipfile

_ckpt = "experiments/hybrid-csharp"
_res = "results/hybrid_csharp_evaluation.json"
_name = "safety_hybrid_csharp"
if os.path.isdir(_ckpt):
    exclude = {"optimizer.pt", "scheduler.pt", "rng_state.pth"}
    with zipfile.ZipFile(f"{_name}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _dirs, fnames in os.walk(_ckpt):
            for fn in fnames:
                if fn in exclude:
                    continue
                fp = os.path.join(root, fn)
                zf.write(fp, os.path.relpath(fp, os.path.dirname(_ckpt)))
    print(f"{_name}.zip -> {os.path.getsize(_name + '.zip') / 1e6:.1f} MB")
    files.download(f"{_name}.zip")
else:
    print("No checkpoint dir at", _ckpt, "-- check the training step above succeeded.")
if os.path.isfile(_res):
    files.download(_res)
else:
    print("No results file at", _res, "-- check the training step above succeeded.")


## Step 7 â€” view results

In [ ]:
import json

for lang in ("java", "csharp"):
    path = f"results/hybrid_{lang}_evaluation.json"
    with open(path) as f:
        print(path)
        print(json.dumps(json.load(f), indent=2))
        print()

## Step 8 â€” download results + checkpoints back to your machine

Colab's local disk is wiped when the runtime ends, so zip up `results/` and `experiments/` and download them now.

In [ ]:
import shutil

shutil.make_archive("hybrid_multilang_output", "zip", WORK_DIR, "results")

files.download("hybrid_multilang_output.zip")